# Training notebook

## Pre-processing

In [ ]:
import datasets as ds
from multipride.preprocessing import load_tokenizer, tokenize_batch
from multipride.evaluation import compute_metrics
from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    AutoModelForSequenceClassification,
)
import torch
import wandb

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0))
print("Current device:", torch.cuda.current_device())

# quick tensor test
x = torch.rand(3, 3).to("cuda")
print("Tensor on GPU:", x.device)

In [ ]:
# loading the dataset
data = ds.load_dataset(
    "csv", data_files="../data/processed/clean_data.csv", split="train"
)
data = data.select_columns(["id", "text", "label"])
print(data)

data = data.cast_column("label", ds.ClassLabel(names=["negative", "positive"]))
print(data.features)

# split the dataset into train and val sets
data_dict = data.train_test_split(
    test_size=0.15, shuffle=True, seed=42, stratify_by_column="label"
)
print(data_dict)

In [ ]:
tokenizer = load_tokenizer()

tokenized_datasets = data_dict.map(
    lambda batch: tokenize_batch(batch, tokenizer), batched=True
)

In [ ]:
train_set = tokenized_datasets["train"]

val_set = tokenized_datasets["test"]

In [ ]:
print(train_set["label"][:10])
print(val_set["label"][:10])

In [ ]:
wandb.login()

In [ ]:
# Initialize Weights & Biases for experiment tracking
wandb.init(project="transformer-fine-tuning", name="first-local-run")

In [ ]:
# Add data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Define the training args
training_args = TrainingArguments(
    output_dir="../results",
    eval_strategy="steps",
    eval_steps=20,
    save_steps=50,
    logging_steps=10,  # Log metrics every 10 steps
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    report_to="wandb",
    fp16=True,  # Send logs to Weights & Biases
    run_name="first_local_run",
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "Musixmatch/umberto-commoncrawl-cased-v1"
)

In [ ]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=train_set,
    eval_dataset=val_set,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
# Close wandb session
wandb.finish()

In [ ]:
trainer.save_model("../model")